# BenignIDS — Setup & Preflight


## Section 0.1 — Sanity Config (Single Source of Truth)

**What this does**
- Declares the only canonical configuration used by all notebooks.

**Must**
- Define **only**: `RANDOM_STATE`, `DATA_PATH`, `TARGET_COL`, `STAGE_ROOT` here.
- Persist a JSON copy (`CANON.json`) so downstream kernels can restore config without redefining it.

**Outputs**
- In-memory constants and created directories (`staging/`, `out/`).
- Files: `./CANON.json` and `STAGE_ROOT/CANON.json` (identical content).


In [1]:
# ======================================================
# Section 0.1 — Sanity Config (Single Source of Truth)
#   • Declares RANDOM_STATE, DATA_PATH, TARGET_COL, STAGE_ROOT
#   • Creates STAGE_ROOT and OUT_ROOT
#   • Persists CANON to ./CANON.json and STAGE_ROOT/CANON.json
# ======================================================
print(">>> Section 0.1 — Sanity Config (Single Source of Truth)")

from pathlib import Path
from types import MappingProxyType
import warnings, re, json

warnings.filterwarnings("ignore")

# ---- Canonical config (edit these as required)
RANDOM_STATE = 42
DATA_PATH    = Path("archive/Payload_data_UNSW.csv")  # source dataset file or folder
TARGET_COL   = "label"                                 # binary target {0,1}
STAGE_ROOT   = Path("staging")                         # single staging root
OUT_ROOT     = Path("out")                             # outputs/reports/models

# ---- Create standard folders
STAGE_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Read-only view for audit
CANON = MappingProxyType({
    "RANDOM_STATE": RANDOM_STATE,
    "DATA_PATH": str(DATA_PATH),
    "TARGET_COL": TARGET_COL,
    "STAGE_ROOT": str(STAGE_ROOT),
    "OUT_ROOT": str(OUT_ROOT),
})
_CANON_NAMES = tuple(CANON.keys())
_CANON_RE = re.compile(r"^\s*(?:{})\s*=\s*".format("|".join(map(re.escape, _CANON_NAMES))))

# ---- Persist CANON for isolated kernels
canon_root = Path("CANON.json")
canon_stage = STAGE_ROOT / "CANON.json"
canon_payload = {k: CANON[k] for k in CANON.keys()}
canon_root.write_text(json.dumps(canon_payload, indent=2), encoding="utf-8")
canon_stage.write_text(json.dumps(canon_payload, indent=2), encoding="utf-8")
print(f"[0.1] Persisted CANON → {canon_root.resolve()}")
print(f"[0.1] Persisted CANON → {canon_stage.resolve()}")

# ---- Status
print(f"[0.1] RANDOM_STATE={RANDOM_STATE}")
print(f"[0.1] DATA_PATH={Path(DATA_PATH).resolve()}")
print(f"[0.1] TARGET_COL={TARGET_COL}")
print(f"[0.1] STAGE_ROOT={Path(STAGE_ROOT).resolve()}")
print(f"[0.1] OUT_ROOT={Path(OUT_ROOT).resolve()}")


>>> Section 0.1 — Sanity Config (Single Source of Truth)
[0.1] Persisted CANON → /Users/max/Library/CloudStorage/OneDrive-Personal/Imperial Pro Cert in Machine Learning and Artificial Intelligence/IMP-PCMLAI-M25-final-project/BenignIDS_Notebooks_v3/CANON.json
[0.1] Persisted CANON → /Users/max/Library/CloudStorage/OneDrive-Personal/Imperial Pro Cert in Machine Learning and Artificial Intelligence/IMP-PCMLAI-M25-final-project/BenignIDS_Notebooks_v3/staging/CANON.json
[0.1] RANDOM_STATE=42
[0.1] DATA_PATH=/Users/max/Library/CloudStorage/OneDrive-Personal/Imperial Pro Cert in Machine Learning and Artificial Intelligence/IMP-PCMLAI-M25-final-project/BenignIDS_Notebooks_v3/archive/Payload_data_UNSW.csv
[0.1] TARGET_COL=label
[0.1] STAGE_ROOT=/Users/max/Library/CloudStorage/OneDrive-Personal/Imperial Pro Cert in Machine Learning and Artificial Intelligence/IMP-PCMLAI-M25-final-project/BenignIDS_Notebooks_v3/staging
[0.1] OUT_ROOT=/Users/max/Library/CloudStorage/OneDrive-Personal/Imperial Pro

## Section 0.2 — Load Data (robust target coercion to binary 0/1)

**What this does**
- Loads `DATA_PATH` → `df` and **coerces the target to strict binary 0/1**
- Accepts a wide set of target encodings (`0/1`, `0.0/1.0`, `True/False`, `benign/malicious`, `attack_cat`, etc.)
- Fails verbosely if unknown labels remain unmapped

**Why**
- Prevents brittle failures later and enforces the project’s data scheme
- Ensures a single, consistent `TARGET_COL` is present before splitting


In [2]:
# ======================================================
# Section 0.2 — Load Data (attack-category fallback mapping)
# Replacement patch: if TARGET_COL contains categorical attack labels
# (e.g. 'generic','exploits','dos', ...) map 'benign'/'normal' -> 0 else -> 1.
# This avoids failure when 'label' is actually an attack category column.
# ======================================================
print(">>> Section 0.2: Load Data (attack-category fallback mapping)")

import os, re, math
import pandas as pd
import numpy as np
from collections import Counter

# Prereqs
assert 'DATA_PATH' in globals() and 'TARGET_COL' in globals(), "Define DATA_PATH and TARGET_COL in 0.1"
assert os.path.exists(DATA_PATH), f"Missing dataset at {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

# Quick probe of the declared target column
if TARGET_COL in df.columns:
    vals = df[TARGET_COL].dropna().astype(str).str.strip()
    uniq = vals.value_counts().head(20)
    print(f"[0.2] {TARGET_COL} unique sample values:\n{uniq.to_string()}\n")
    # If values are numeric-like 0/1 -> keep handled earlier; otherwise treat as categorical attack labels
    non_numeric_frac = (~vals.str.match(r'^[01]$')).mean() if len(vals) else 0.0
    if non_numeric_frac > 0.01:
        # Heuristic: map explicit benign tokens to 0; everything else -> 1
        benign_tokens = {"benign","normal","noattack","none","safe","clean","background","0","false"}
        def map_attack_cat(v):
            if pd.isna(v):
                return np.nan
            s = str(v).strip().lower()
            if s in benign_tokens:
                return 0
            # treat numeric strings '0'/'1' explicitly
            if s in {'0','1'}:
                return int(s)
            # otherwise it's an attack category label -> malicious (=1)
            return 1
        mapped = vals.map(map_attack_cat)
        if mapped.isna().any():
            print("[0.2] Warning: some target values could not be mapped; showing examples:")
            print(vals[mapped.isna()].value_counts().head(20).to_string())
            # fall back to raising so user can inspect if desired
            raise AssertionError("Some TARGET_COL values remain unmapped; inspect data or extend benign_tokens.")
        df[TARGET_COL] = mapped.astype(int)
        print(f"[0.2] Mapped attack-category labels -> binary 0/1 using heuristic (benign_tokens).")
    else:
        # proceed with robust coercion from earlier logic (numeric/strings mapping)
        def _lower_str(x):
            if pd.isna(x):
                return None
            return str(x).strip().lower()
        def try_numeric_coerce(s):
            if pd.api.types.is_bool_dtype(s):
                return s.astype(int)
            if pd.api.types.is_integer_dtype(s):
                u = set(s.dropna().unique().tolist())
                if u.issubset({0,1}):
                    return s.astype(int)
            if pd.api.types.is_float_dtype(s):
                u = set(np.unique(s.dropna().values))
                if u.issubset({0.0,1.0}):
                    return s.astype(int)
            # else map strings
            mapped = []
            for v in s.values:
                if pd.isna(v):
                    mapped.append(np.nan); continue
                t = _lower_str(v)
                if t in {"0","1"}:
                    mapped.append(int(t)); continue
                if t in {"false","no","neg","benign","normal","noattack","none","safe","clean"}:
                    mapped.append(0); continue
                if t in {"true","yes","pos","malicious","attack","intrusion","malware"}:
                    mapped.append(1); continue
                # unknown -> nan
                mapped.append(np.nan)
            out = pd.Series(mapped, index=s.index, dtype="float64")
            if out.isna().any():
                # leave original handling to earlier fallback attempts
                raise AssertionError("Direct coercion failed; will try fallbacks.")
            return out.astype(int)
        # attempt
        try:
            df[TARGET_COL] = try_numeric_coerce(df[TARGET_COL])
            print(f"[0.2] Coerced '{TARGET_COL}' to binary 0/1 via string/numeric mapping.")
        except AssertionError:
            # allow subsequent fallbacks (label_str / attack_cat) to be used by the notebook flow
            print(f"[0.2] Direct coercion of '{TARGET_COL}' not fully successful; will try fallbacks next.")

# If TARGET_COL not present, downstream code should already attempt label_str/attack_cat fallbacks.
if TARGET_COL not in df.columns:
    # leave to existing code path that handles label_str / attack_cat
    pass

# Final assert: ensure binary target exists or inform explicitly
if TARGET_COL in df.columns:
    if not df[TARGET_COL].isin([0,1]).all():
        bad = df[TARGET_COL].value_counts().head(20)
        raise AssertionError(f"After mapping, some '{TARGET_COL}' values are not 0/1. Top examples:\n{bad.to_string()}")
    print(f"[0.2] Final check: '{TARGET_COL}' is binary 0/1 and ready.")
else:
    raise AssertionError("TARGET_COL not present and no fallbacks created it. Ensure 'label_str' or 'attack_cat' exist.")


>>> Section 0.2: Load Data (attack-category fallback mapping)


[0.2] label unique sample values:
label
normal            21000
generic           17580
exploits          13992
fuzzers           12722
reconnaissance     7562
dos                3397
backdoor           1239
analysis           1208
shellcode          1088
worms                93

[0.2] Mapped attack-category labels -> binary 0/1 using heuristic (benign_tokens).
[0.2] Final check: 'label' is binary 0/1 and ready.


## Section 0.3 — Target Audit & Canonicalisation
- Derives/validates binary `label` and folds `payload_byte_*` → `payload`.


In [3]:
# ======================================================
# Section 0.3 — Target Audit & Canonicalisation
# ======================================================
print(">>> Section 0.3 — Target Audit & Canonicalisation")

import re
import numpy as np
import pandas as pd

assert 'df' in globals(), "[0.3] df not loaded; run 0.2 first."
assert 'TARGET_COL' in globals(), "[0.3] TARGET_COL not configured; run 0.1 first."

if TARGET_COL not in df.columns:
    label_str_map = {"benign":0, "normal":0, "noattack":0, "false":0, "neg":0,
                     "malicious":1, "attack":1, "true":1, "pos":1}
    def _attack_cat_to_label(v):
        if pd.isna(v): return np.nan
        return 0 if str(v).strip().lower() == "benign" else 1
    if "label_str" in df.columns:
        df[TARGET_COL] = df["label_str"].astype(str).str.strip().str.lower().map(label_str_map)
        print("[0.3] Derived TARGET_COL from label_str mapping.")
    elif "attack_cat" in df.columns:
        df[TARGET_COL] = df["attack_cat"].apply(_attack_cat_to_label)
        print("[0.3] Derived TARGET_COL from attack_cat mapping.")
    else:
        raise AssertionError("[0.3] Cannot derive TARGET_COL: no label, label_str, or attack_cat present.")

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").astype("Int64").fillna(0).astype(int)
uniq = set(df[TARGET_COL].unique().tolist())
assert uniq.issubset({0,1}), f"[0.3] TARGET_COL must be binary in {{0,1}}; got {uniq}"
print(f"[0.3] TARGET_COL distribution: {df[TARGET_COL].value_counts(dropna=False).to_dict()}")

payload_re = re.compile(r"^payload_byte_(\d+)$")
payload_cols = [c for c in df.columns if payload_re.match(c)]
if payload_cols:
    payload_cols_sorted = sorted(payload_cols, key=lambda c: int(payload_re.match(c).group(1)))
    df["payload"] = df[payload_cols_sorted].to_numpy().tolist()
    df.drop(columns=payload_cols_sorted, inplace=True, errors="ignore")
    print(f"[0.3] Folded {len(payload_cols_sorted)} payload byte columns → 'payload' and dropped originals.")
else:
    print("[0.3] No payload_byte_* columns detected — nothing to fold.")

print("[0.3] Target audit & canonicalisation complete.")


>>> Section 0.3 — Target Audit & Canonicalisation
[0.3] TARGET_COL distribution: {1: 58881, 0: 21000}


[0.3] Folded 1500 payload byte columns → 'payload' and dropped originals.
[0.3] Target audit & canonicalisation complete.


## Section 0.4 — Split & Preprocessing (minimal, payload-first pipeline)

**What this does**
- Creates stratified **train/val/test** splits: `X_train`, `X_val`, `X_test` and `y_*`
- Keeps all original feature columns (including `payload_byte_*`) for Section 2.1 folding

**Why**
- Establishes reproducible splits with `RANDOM_STATE`
- Defers feature engineering to Section 2.1 so only `payload` is used for model training later


In [4]:
# ======================================================
# Section 0.4 — Split & Preprocessing (minimal, payload-first pipeline)
#   • Stratified train/val/test
#   • Preserve raw columns for Section 2.1 folding (payload only used in training later)
# ======================================================
print(">>> Section 0.4: Split & Preprocessing")

from sklearn.model_selection import train_test_split

assert 'RANDOM_STATE' in globals(), "Define RANDOM_STATE in 0.1"
assert 'df' in globals() and TARGET_COL in df.columns, "Run 0.2 first; ensure df and TARGET_COL exist"

# Separate features/target; do NOT engineer features here (payload handled in 2.1)
y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

# Two-step split: train/holdout → val/test
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.5, random_state=RANDOM_STATE, stratify=y_hold
)

# Minimal integrity checks
assert len(X_train) and len(X_val) and len(X_test), "Empty split detected"
print(f"[0.4] Split shapes → train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")
print(f"[0.4] Class balance (train) → pos={y_train.sum()} / n={len(y_train)}")


>>> Section 0.4: Split & Preprocessing


[0.4] Split shapes → train=(55916, 5), val=(11982, 5), test=(11983, 5)
[0.4] Class balance (train) → pos=41216 / n=55916


## Section 0.6 — Save & Restore Staging Training Data
- Saves `X_train/y_train`, `X_val/y_val`, `X_test/y_test` to `STAGE_ROOT/splits/`.


In [5]:
# ======================================================
# Section 0.6 — Save & Restore Staging Training Data
# ======================================================
print(">>> Section 0.6 — Save & Restore Staging Training Data")

import joblib
from pathlib import Path

assert 'STAGE_ROOT' in globals(), "[0.6] STAGE_ROOT not defined in config (0.1)."
required = ['X_train','y_train','X_val','y_val','X_test','y_test']
missing = [v for v in required if v not in globals()]
assert not missing, f"[0.6] Missing variables from 0.4: {missing}"

splits_dir = Path(STAGE_ROOT) / 'splits'
splits_dir.mkdir(parents=True, exist_ok=True)

for name in required:
    joblib.dump(globals()[name], splits_dir / f"{name}.joblib")
    print(f"[0.6] Saved {name} → {splits_dir / (name + '.joblib')}")

print("[0.6] Staging save complete.")


>>> Section 0.6 — Save & Restore Staging Training Data


[0.6] Saved X_train → staging/splits/X_train.joblib
[0.6] Saved y_train → staging/splits/y_train.joblib


[0.6] Saved X_val → staging/splits/X_val.joblib
[0.6] Saved y_val → staging/splits/y_val.joblib


[0.6] Saved X_test → staging/splits/X_test.joblib
[0.6] Saved y_test → staging/splits/y_test.joblib
[0.6] Staging save complete.


## Section 0.035 — Canonical Guard: `drop_like` Purge Scanner

In [6]:
# ======================================================
# Section 0.035 — Canonical Guard: `drop_like` Purge Scanner
# ======================================================
print(">>> Section 0.035 — Canonical Guard: `drop_like` Purge Scanner")

import inspect, json, re
from pathlib import Path

hits = []
for name, obj in list(globals().items()):
    if callable(obj):
        try: src = inspect.getsource(obj)
        except Exception: continue
        if "drop_like" in src: hits.append(("in-memory", name, "<callable>", None))

root = Path(".")
file_patterns = (r".*\.ipynb$", r".*\.py$", r".*\.md$")
rxs = [re.compile(pat) for pat in file_patterns]
def file_wanted(p: Path) -> bool: return any(rx.match(p.name) for rx in rxs)

for p in root.rglob("*"):
    if not p.is_file() or not file_wanted(p): continue
    try:
        if p.suffix == ".ipynb":
            data = json.loads(p.read_text(encoding="utf-8"))
            for cell_idx, cell in enumerate(data.get("cells", [])):
                if cell.get("cell_type") == "code":
                    src = "".join(cell.get("source", []))
                    if "drop_like" in src: hits.append(("file", str(p), "ipynb_cell", cell_idx+1))
        else:
            txt = p.read_text(encoding="utf-8", errors="ignore")
            if "drop_like" in txt: hits.append(("file", str(p), p.suffix, None))
    except Exception as e:
        print(f"[0.035][warn] Skipped {p} due to: {e!r}")

if not hits:
    print("[0.035][guard] No 'drop_like' references found in memory or files.")
else:
    print("[0.035][guard] Found 'drop_like' references:")
    for kind, loc, extra, idx in hits:
        where = f"{loc}"; 
        if idx is not None: where += f" (code cell #{idx})"
        print(f"  - {kind:9s} | {where:60s} | {extra}")


>>> Section 0.035 — Canonical Guard: `drop_like` Purge Scanner
[0.035][guard] Found 'drop_like' references:
  - file      | 01-Setup_Preflight.ipynb (code cell #13)                     | ipynb_cell
  - file      | executed_from_driver/01-Setup_Preflight.ipynb (code cell #13) | ipynb_cell
